In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dropout, Layer
from tensorflow.keras.layers import Embedding, Input, GlobalAveragePooling1D, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import pandas as pd
import warnings
import os
import pickle
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)
from itertools import chain
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, auc, confusion_matrix,roc_auc_score
import matplotlib.pyplot as plt
import plotly.figure_factory as ff

In [2]:
import nltk
#nltk.download("punkt")
nltk.download('stopwords')
from nltk.lm.preprocessing import pad_sequence
from nltk.tokenize import word_tokenize 
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tommy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
import wandb

In [4]:
file = "customer_support"

In [5]:
labels_column = "issue_area"
text_column = "conversation"

In [6]:
wandb.init(project=file, sync_tensorboard=True, save_code = True)
wandb.config["more"] = "custom"

wandb: Currently logged in as: tommytsuma7. Use `wandb login --relogin` to force relogin


In [49]:
from wandb.integration.keras import WandbCallback, WandbMetricsLogger , WandbModelCheckpoint 

In [8]:
df = pd.read_csv(f"D://CoreOutline/data/{file}.csv")

In [9]:
df

,Unnamed: 0,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
0,0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or ...,Mobile Number and Email Verification -> Verifi...,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
1,1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox customer...
2,2,Cancellations and returns,Replacement and Return Process,Inability to click the 'Cancel' button,Replacement and Return Process -> Inability to...,neutral,Appliances,Juicer/Mixer/Grinder,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...
3,3,Login and Account,Login Issues and Error Messages,Error message regarding exceeded attempts to e...,Login Issues and Error Messages -> Error messa...,neutral,Appliances,Water Purifier,less,inexperienced,"may struggle with ambiguous queries, rely on c...","Customer: Hi, I am facing an issue while loggi..."
4,4,Order,Order Delivery Issues,Delivery not attempted again,Order Delivery Issues -> Delivery not attempte...,negative,Electronics,Bp Monitor,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for contacting BrownBox custo...
...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,Cancellations and returns,Return and Exchange,Package open or tampered on delivery,Return and Exchange -> Package open or tampere...,negative,Electronics,Mobile,medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
996,996,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Men/Women/Kids,Backpack,medium,junior,"handles customer inquiries independently, poss...","Customer: Hi, I received an email from BrownBo..."
997,997,Warranty,Warranty Terms and Changes,Warranty mismatch between the website and the ...,Warranty Terms and Changes -> Warranty mismatc...,negative,Appliances,Water Purifier,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
998,998,Cancellations and returns,Return and Exchange,Checking the status of a refund,Return and Exchange -> Checking the status of ...,neutral,Appliances,Wet Grinder,medium,junior,"handles customer inquiries independently, poss...","Customer: Hi, I would like to check the status..."


In [10]:
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential(
            [Dense(ff_dim, activation="relu"), 
             Dense(embed_dim),]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

In [11]:
class TokenAndPositionEmbedding(Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [12]:
vocab_size = 20000  # Only consider the top 20k words
maxlen = 518  # Only consider the first 200 words of each movie review

(x_train, y_train), (x_val, y_val) = imdb.load_data(num_words=vocab_size)
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")

25000 Training sequences
25000 Validation sequences


In [13]:
x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_val = tf.keras.preprocessing.sequence.pad_sequences(x_val, maxlen=maxlen)

In [14]:
embed_dim = 32  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network inside transformer

inputs = Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(20, activation="relu")(x)
x = Dropout(0.1)(x)
outputs = Dense(len(df[labels_column].unique()), activation="softmax")(x)



In [15]:
def remove_punctuation(x):
    return "".join([i for i in x if i not in string.punctuation])

In [16]:
df['no_punctuation_conversation'] = df[text_column].apply(remove_punctuation)

In [17]:
df['lowercase_conversation'] = df.no_punctuation_conversation.str.lower()

In [18]:
df['tokenized_conversation'] = [word_tokenize(i) for i in df['lowercase_conversation']]

In [19]:
def remove_stopwords(tokens):
    stopwords = nltk.corpus.stopwords.words('english')
    return [token for token in tokens if token not in stopwords]

In [20]:
df['no_stopwords_conversation'] = df['tokenized_conversation'].apply(remove_stopwords)

In [21]:
df['no_stopwords_conversation']

0      [agent, thank, calling, brownbox, customer, su...
1      [agent, thank, calling, brownbox, customer, su...
2      [agent, thank, calling, brownbox, customer, su...
3      [customer, hi, facing, issue, logging, account...
4      [agent, thank, contacting, brownbox, customer,...
                             ...                        
995    [agent, thank, calling, brownbox, customer, su...
996    [customer, hi, received, email, brownbox, stat...
997    [agent, thank, calling, brownbox, customer, su...
998    [customer, hi, would, like, check, status, ref...
999    [customer, hi, calling, received, mobile, phon...
Name: no_stopwords_conversation, Length: 1000, dtype: object

In [22]:
words = np.array(df['no_stopwords_conversation'])

In [23]:
words = list(chain(*words))

In [24]:
le = LabelEncoder()
df['issue_area_le'] = le.fit_transform(df[labels_column])

In [25]:
def word_map_tokenizer(text_arr):
    tokenizer = Tokenizer(num_words = 512, oov_token = '<00V>')
    tokenizer.fit_on_texts(text_arr)
    return tokenizer

In [26]:
tokenizer = word_map_tokenizer(words)

In [27]:
pad_token = 0  # The token to use for padding
max_length = max(len(seq) for seq in df['no_stopwords_conversation']) 

In [28]:
max_length

518

In [29]:
df['conversation_encoded'] = [list(chain(*tokenizer.texts_to_sequences(i))) for i in df['no_stopwords_conversation'] ]

In [30]:
df['conversation_encoded'] 

0      [3, 4, 60, 17, 2, 40, 18, 169, 11, 9, 36, 2, 3...
1      [3, 4, 60, 17, 2, 40, 18, 91, 11, 9, 36, 2, 38...
2      [3, 4, 60, 17, 2, 40, 18, 35, 11, 9, 36, 2, 38...
3      [2, 38, 225, 53, 298, 31, 499, 354, 428, 1, 1,...
4      [3, 4, 93, 17, 2, 40, 18, 35, 9, 36, 2, 38, 35...
                             ...                        
995    [3, 4, 60, 17, 2, 40, 18, 35, 11, 9, 36, 2, 38...
996    [2, 38, 61, 12, 17, 1, 39, 277, 127, 303, 94, ...
997    [3, 4, 60, 17, 2, 40, 18, 35, 11, 9, 36, 2, 38...
998    [2, 38, 67, 73, 13, 90, 37, 1, 1, 219, 70, 212...
999    [2, 38, 60, 61, 71, 124, 182, 1, 118, 146, 70,...
Name: conversation_encoded, Length: 1000, dtype: object

In [31]:
[len(i) for i in df['conversation_encoded'] if len(i) > 500]

[518, 509]

In [32]:
df['conversation_padded'] = [list(pad_sequence(i, n=(519-len(i)), pad_left=True, left_pad_symbol=pad_token)) for i in df['conversation_encoded']]

In [33]:
df['conversation_padded']

0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                             ...                        
995    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
996    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
997    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
998    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
999    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: conversation_padded, Length: 1000, dtype: object

In [34]:
[len(i) for i in df['conversation_padded']]

[518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518,
 518

In [35]:
X = np.array(pd.DataFrame([ i for i in df['conversation_padded']]))
y=  np.array(df['issue_area_le'])

In [36]:
X_train, X_test_val, y_train, y_test_val = train_test_split(X,y)

In [37]:
X_test, X_val, y_test, y_val = train_test_split(X_test_val,y_test_val)

In [38]:
len(X_test)

187

In [39]:
np.shape(y_train)

(750,)

In [40]:
np.shape(X_train)

(750, 518)

In [61]:
callbacks = []

In [64]:
callbacks.append(WandbMetricsLogger())

In [65]:
earlystopping = EarlyStopping(monitor="val_accuracy",
                             patience=10,
                             min_delta=0,
                             mode='min',
                             restore_best_weights=False,
                             baseline=None,
                             verbose=0)
callbacks.append(earlystopping)

In [66]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy',
                                                 factor=0.5,
                                                 patience=30,
                                                 min_lr=0.000001,
                                                 cooldown=5)

callbacks.append(reduce_lr)

In [67]:
checkpoint_path = "training_2/cp-{epoch:04d}.keras"
checkpoint_dir = os.path.dirname(checkpoint_path)

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path, 
    verbose=1, 
    save_weights_only=False
    )

callbacks.append(cp_callback)

In [68]:
model = Model(inputs=inputs, outputs=outputs)


In [69]:
model.compile(optimizer=Adam(learning_rate=0.01), loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [70]:
history = model.fit(X_train, y_train, 
                    batch_size=32, epochs=200, 
                    validation_data=(X_val, y_val),
                    callbacks=callbacks
                   )

Epoch 1/200
24/24 ━━━━━━━━━━━━━━━━━━━━ 1:53 5s/step - accuracy: 0.7500 - loss: 0.518 ━━━━━━━━━━━━━━━━━━━━ 2s 132ms/step - accuracy: 0.7656 - loss: 0.51 ━━━━━━━━━━━━━━━━━━━━ 2s 131ms/step - accuracy: 0.7674 - loss: 0.53 ━━━━━━━━━━━━━━━━━━━━ 2s 129ms/step - accuracy: 0.7572 - loss: 0.58 ━━━━━━━━━━━━━━━━━━━━ 2s 127ms/step - accuracy: 0.7545 - loss: 0.60 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.7537 - loss: 0.61 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.7545 - loss: 0.61 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.7564 - loss: 0.61 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.7599 - loss: 0.60 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.7630 - loss: 0.60 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.7659 - loss: 0.60 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.7687 - loss: 0.59 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.7717 - loss: 0.59 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.7738 - loss: 0.58 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accurac

In [ ]:
for accuracy, loss in zip(history.history['val_accuracy'], history.history['val_loss']):
    wandb.log({"accuracy": accuracy, "loss": loss})
wandb.finish()

In [ ]:
y_preds = model.predict(X_test)

In [ ]:
y_preds = [ np.where(i==max(i)) for i in y_preds]

In [ ]:
y_preds = [i[0] for i in list(chain(*y_preds))]

In [ ]:
model.save_weights(f"D://CoreOutline/transformer/models/{file}.weights.h5")
results = model.evaluate(X_val, y_val, verbose=2)

for name, value in zip(model.metrics_names, results):
    print("%s: %.3f" % (name, value))

In [ ]:
print(classification_report(y_test, y_preds))

In [ ]:
cf_matrix = confusion_matrix(y_test, y_preds)

In [ ]:
y_preds_ = le.inverse_transform(y_preds)
y_test_ = le.inverse_transform(y_test)

In [ ]:
def plotPredictionConfusionMatrix(df, columns):
    fig = ff.create_annotated_heatmap(z=np.array(df),x=[i for i in columns], y=[i for i in columns], colorscale='blues', showscale=False, reversescale=False)
    fig['layout']['xaxis'].update(side='bottom', title='Actual')
    fig['layout']['yaxis'].update(side='left', title='Predicted')
    fig.update_layout(title=f'Prediction Confusion Matrix Heatmap', width=1000, height=1000)
    fig.write_image(f"D://CoreOutline/transformer/visualizations/{file}_heatmap.png")

In [ ]:
plotPredictionConfusionMatrix(cf_matrix, np.unique(y_preds_))

In [ ]:
custom_objects = {"TokenAndPositionEmbedding": TokenAndPositionEmbedding, "TransformerBlock":TransformerBlock}

In [ ]:
#pickle.dump(custom_objects, open("D://CoreOutline/transformer/models/custom_object.dict","wb"))

In [ ]:
#model.save_model("D://CoreOutline/transformer/models/text_clf.h5")

In [ ]:
model

In [ ]:
pickle.dump(history, open(f"D://CoreOutline/transformer/history/{file}_history","wb"))

In [ ]:
pickle.dump(tokenizer, open(f"D://CoreOutline/transformer/tokens/{file}_tokenizer","wb"))

In [ ]:
plt.plot(history.history['val_accuracy'])
plt.ylabel('Validation Accuracy')
plt.xlabel('Epochs')
plt.savefig(f"D://CoreOutline/transformer/visualizations/{file}_accuracy_line_plot.png")
plt.clf()

In [ ]:
plt.plot(history.history['val_loss'])
plt.ylabel('Validation Loss')
plt.xlabel('Epochs')
plt.savefig(f"D://CoreOutline/transformer/visualizations/{file}_loss_line_plot.png")
plt.clf()